# Floor Plan OCR — Tesseract replacement for `ocr_floor_plan()`

Reads room names and sizes from floor plan images using local Tesseract OCR.  
Target: drop-in replacement for the Haiku vision call, zero API cost.

**Images**: `training/labeled/floor_plan/` — 517 labeled images  
**Output format**: same as Haiku — one room per line: `Room: Xm²`

In [ ]:
import subprocess, sys, shutil

# ── 1. Check Tesseract binary ──────────────────────────────────────────────
tess = shutil.which('tesseract')
if not tess:
    print('ERROR: tesseract not found. Install: brew install tesseract tesseract-lang')
else:
    v = subprocess.check_output(['tesseract', '--version'], stderr=subprocess.STDOUT).decode().splitlines()[0]
    print('Tesseract:', v)

# ── 2. Check Japanese language data ───────────────────────────────────────
langs = subprocess.check_output(['tesseract', '--list-langs'], stderr=subprocess.STDOUT).decode()
if 'jpn' not in langs:
    print()
    print('Japanese language data not found.')
    print('Install: brew install tesseract-lang')
    print('Then re-run this cell.')
else:
    print('Languages: jpn + jpn_vert available ✓')

# ── 3. Python deps ──────────────────────────────────────────────────────────
try:
    import pytesseract
    print('pytesseract:', pytesseract.__version__)
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytesseract'])
    import pytesseract

from PIL import Image, ImageFilter, ImageOps
import numpy as np
import re
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import textwrap

print('All imports OK')

In [ ]:
# ── Load images ────────────────────────────────────────────────────────────
LABELED_DIR = Path('training/labeled/floor_plan')
image_paths = sorted(LABELED_DIR.glob('*.jpg')) + sorted(LABELED_DIR.glob('*.png'))
print(f'Found {len(image_paths)} floor plan images')

# Quick sanity — show a 3×3 grid of samples
sample = image_paths[:9]
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for ax, p in zip(axes.flat, sample):
    ax.imshow(Image.open(p))
    ax.set_title(p.name[:30], fontsize=7)
    ax.axis('off')
plt.suptitle('Sample floor plan images', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────────
# Floor plans: light background, small black text, sometimes coloured fills.
# Strategy: grayscale → upscale 2× → adaptive threshold

def preprocess(path, scale=2):
    img = Image.open(path).convert('RGB')
    gray = img.convert('L')
    # Upscale — bigger pixels = easier for Tesseract
    w, h = gray.size
    gray = gray.resize((w * scale, h * scale), Image.LANCZOS)
    # Otsu-style threshold via numpy
    arr = np.array(gray, dtype=np.float32)
    hist, bins = np.histogram(arr.ravel(), bins=256, range=(0, 256))
    # Otsu threshold
    total = arr.size
    w0, sum_total, sum_b = 0, np.dot(np.arange(256), hist), 0
    var_max, thresh = 0, 128
    for i in range(256):
        w0 += hist[i]
        if w0 == 0: continue
        w1 = total - w0
        if w1 == 0: break
        sum_b += i * hist[i]
        m0, m1 = sum_b / w0, (sum_total - sum_b) / w1
        var_between = w0 * w1 * (m0 - m1) ** 2
        if var_between > var_max:
            var_max, thresh = var_between, i
    binary = (arr > thresh).astype(np.uint8) * 255
    return Image.fromarray(binary)


# Show raw vs preprocessed side-by-side for 4 images
fig, axes = plt.subplots(4, 2, figsize=(12, 14))
for i, p in enumerate(image_paths[:4]):
    axes[i, 0].imshow(Image.open(p))
    axes[i, 0].set_title('Original', fontsize=8)
    axes[i, 0].axis('off')
    axes[i, 1].imshow(preprocess(p), cmap='gray')
    axes[i, 1].set_title('Preprocessed (2× + Otsu)', fontsize=8)
    axes[i, 1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Tesseract OCR ──────────────────────────────────────────────────────────
# PSM modes relevant for floor plans:
#   3 = fully auto (default)
#   6 = single uniform block of text
#   11 = sparse text (best for scattered labels)
#   12 = sparse text with OSD

LANG = 'jpn+eng'

def ocr_raw(path, psm=11, scale=2):
    img = preprocess(path, scale=scale)
    config = f'--lang {LANG} --psm {psm} --oem 3'
    return pytesseract.image_to_string(img, config=config)


# ── Parser ─────────────────────────────────────────────────────────────────
# Extracts lines like: 洋室1: 6.0m² or LDK: 14.5㎡ or 洗面所 2.5

ROOM_PATTERN = re.compile(
    r'([\u3000-\u9FFF\uF900-\uFAFF\w]{1,10}[0-9]?)'
    r'[\s:：\t]+'
    r'([0-9]+\.?[0-9]*)'
    r'\s*[mｍ㎡²]'
)

def parse_rooms(text):
    matches = ROOM_PATTERN.findall(text)
    lines = [f'{room}: {size}m²' for room, size in matches if float(size) > 0]
    return '\n'.join(lines) if lines else None


# Quick test on a few images with different PSM modes
test_paths = image_paths[:6]
for p in test_paths:
    print(f'\n── {p.name} ──')
    for psm in [6, 11, 12]:
        raw = ocr_raw(p, psm=psm)
        parsed = parse_rooms(raw)
        n_rooms = len(parsed.splitlines()) if parsed else 0
        print(f'  PSM {psm:2d}: {n_rooms} rooms found | raw chars: {len(raw.strip())}')
    best = ocr_raw(p, psm=11)
    parsed = parse_rooms(best)
    print('  Parsed:', parsed or '(none)')

In [ ]:
# ── Visual evaluation: image + OCR text side by side ──────────────────────
N_SHOW = 16  # adjust as needed

fig = plt.figure(figsize=(20, N_SHOW * 2.5))
gs = gridspec.GridSpec(N_SHOW, 2, width_ratios=[1, 1], figure=fig)

results = []
for i, p in enumerate(image_paths[:N_SHOW]):
    raw = ocr_raw(p, psm=11)
    parsed = parse_rooms(raw)
    results.append({'path': p, 'raw': raw, 'parsed': parsed})

    ax_img = fig.add_subplot(gs[i, 0])
    ax_txt = fig.add_subplot(gs[i, 1])

    ax_img.imshow(Image.open(p))
    ax_img.axis('off')
    ax_img.set_title(p.name[:25], fontsize=7, pad=2)

    display_text = parsed or '(no rooms detected)'
    ax_txt.text(0.02, 0.98, display_text, transform=ax_txt.transAxes,
                va='top', ha='left', fontsize=8,
                fontfamily='monospace',
                color='#1a1a1a' if parsed else '#cc4444',
                wrap=True)
    ax_txt.axis('off')

plt.suptitle('Floor Plan OCR — image vs extracted rooms', fontsize=13, y=1.002)
plt.tight_layout()
plt.show()

In [ ]:
# ── Batch evaluation on all 517 images ────────────────────────────────────
from tqdm.auto import tqdm

all_results = []
for p in tqdm(image_paths, desc='OCR'):
    try:
        raw = ocr_raw(p, psm=11)
        parsed = parse_rooms(raw)
        n_rooms = len(parsed.splitlines()) if parsed else 0
        all_results.append({'path': p, 'raw': raw, 'parsed': parsed, 'n_rooms': n_rooms})
    except Exception as e:
        all_results.append({'path': p, 'raw': '', 'parsed': None, 'n_rooms': 0, 'error': str(e)})

# Stats
n_total = len(all_results)
n_any   = sum(1 for r in all_results if r['parsed'])
n_2plus = sum(1 for r in all_results if r['n_rooms'] >= 2)
avg_rooms = np.mean([r['n_rooms'] for r in all_results if r['parsed']])

print(f'Total images:       {n_total}')
print(f'Any rooms found:    {n_any}  ({100*n_any/n_total:.1f}%)')
print(f'≥2 rooms found:     {n_2plus} ({100*n_2plus/n_total:.1f}%)')
print(f'Avg rooms (when >0): {avg_rooms:.1f}')

# Room count distribution
room_counts = [r['n_rooms'] for r in all_results]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(room_counts, bins=range(0, max(room_counts)+2), align='left', rwidth=0.8, color='#1976d2')
ax.set_xlabel('Rooms detected per image')
ax.set_ylabel('Count')
ax.set_title('OCR room detection distribution')
plt.tight_layout()
plt.show()

In [ ]:
# ── Failure cases — images with no rooms detected ─────────────────────────
failures = [r for r in all_results if not r['parsed']]
print(f'Failures: {len(failures)}/{n_total}')

N_FAIL = min(12, len(failures))
if N_FAIL > 0:
    cols = 4
    rows = (N_FAIL + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.5))
    for ax, r in zip(axes.flat, failures[:N_FAIL]):
        ax.imshow(Image.open(r['path']))
        ax.set_title(r['path'].name[:28], fontsize=7)
        ax.axis('off')
    for ax in axes.flat[N_FAIL:]:
        ax.axis('off')
    plt.suptitle('Images with 0 rooms detected (failures)', fontsize=11)
    plt.tight_layout()
    plt.show()

# Show raw OCR of first few failures
for r in failures[:3]:
    print(f'\n{r["path"].name}')
    print(repr(r['raw'][:300]))

In [ ]:
# ── Best results — most rooms extracted ───────────────────────────────────
best = sorted([r for r in all_results if r['parsed']], key=lambda r: -r['n_rooms'])[:6]
for r in best:
    print(f'{r["path"].name} → {r["n_rooms"]} rooms')
    print(r['parsed'])
    print()

In [ ]:
# ── Drop-in replacement function ──────────────────────────────────────────
# Paste this into image_pipeline.py to replace ocr_floor_plan()

REPLACEMENT_CODE = '''
def ocr_floor_plan(image_path: str) -> str | None:
    """
    Extract room names and sizes from a floor plan image using local Tesseract.
    Returns formatted text (one room per line: "Room: Xm²") or None.
    """
    import re
    import numpy as np
    try:
        import pytesseract
        from PIL import Image
    except ImportError:
        log.warning("pytesseract or Pillow not installed — skipping OCR")
        return None

    ROOM_PATTERN = re.compile(
        r"([\u3000-\u9FFF\uF900-\uFAFFa-zA-Z][\u3000-\u9FFF\uF900-\uFAFF\w]{0,9}[0-9]?)"
        r"[\\s:：\\t]+"
        r"([0-9]+\\.?[0-9]*)"
        r"\\s*[mｍ㎡²]"
    )

    try:
        img = Image.open(image_path).convert("L")
        w, h = img.size
        img = img.resize((w * 2, h * 2), Image.LANCZOS)
        arr = np.array(img, dtype=np.float32)
        # Otsu threshold
        hist, _ = np.histogram(arr.ravel(), bins=256, range=(0, 256))
        total, w0, sum_total, sum_b, var_max, thresh = arr.size, 0, np.dot(np.arange(256), hist), 0, 0, 128
        for i in range(256):
            w0 += hist[i]; w1 = total - w0
            if w0 == 0 or w1 == 0: continue
            sum_b += i * hist[i]
            m0, m1 = sum_b / w0, (sum_total - sum_b) / w1
            v = w0 * w1 * (m0 - m1) ** 2
            if v > var_max: var_max, thresh = v, i
        proc = Image.fromarray((arr > thresh).astype(np.uint8) * 255)
        raw = pytesseract.image_to_string(proc, config="--lang jpn+eng --psm 11 --oem 3")
        matches = ROOM_PATTERN.findall(raw)
        lines = [f"{room}: {size}m²" for room, size in matches if float(size) > 0]
        return "\\n".join(lines) if lines else None
    except Exception as e:
        log.warning(f"OCR failed for {image_path}: {e}")
        return None
'''

print(REPLACEMENT_CODE)